# Week 19: Real-time GIS: WebSockets and streaming TLEs

**Track:** Mission GIS Engineer (Advanced)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/19/](https://launchdetect.com/academy/week/19/)  
**Track index:** [https://launchdetect.com/academy/mission-gis-engineer/](https://launchdetect.com/academy/mission-gis-engineer/)

---

_Real-time isn't optional in serious space GIS. This week you build the WebSocket pipeline that streams live satellite positions to a browser, throttling and reconnecting._


## Why this week matters

Real-time isn't optional for launch tracking. A 5-minute delay is the difference between 'live' and 'historical'. WebSockets are how every modern real-time GIS works.


## Learning objectives

By the end of this lab you will be able to:

- Build a WebSocket server in Python (FastAPI / Socket.IO)
- Stream live satellite position updates to a browser
- Throttle updates to keep the browser responsive
- Handle disconnects and reconnects gracefully


## Setup — and why these dependencies

This lab uses: `leafmap[common] requests websockets`. Run the cell below once; Colab installs into the runtime.


In [ ]:
# Install required packages
!pip install -q leafmap[common] requests websockets


## The approach (and why)

Build a FastAPI WebSocket server that streams synthetic positions at 1 Hz. Once that's working, swap in real skyfield-propagated positions. The protocol is the easy part; the throttling and back-pressure handling is where production complexity lives.


In [ ]:
# Week 19: real-time WebSocket streaming — server skeleton.
# Run the server in one cell, connect from another (or a browser).

server_code = """
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
import asyncio, json, random
app = FastAPI()

@app.websocket("/ws/positions")
async def positions(ws: WebSocket):
    await ws.accept()
    try:
        while True:
            payload = [{"sat_id": i, "lat": random.uniform(-60, 60),
                        "lon": random.uniform(-180, 180)}
                       for i in range(10)]
            await ws.send_text(json.dumps(payload))
            await asyncio.sleep(1.0)
    except WebSocketDisconnect:
        pass
"""

with open("ws_server.py", "w") as f:
    f.write(server_code)
print("Wrote ws_server.py")
print("Run: uvicorn ws_server:app --port 8000")
print("Then connect from a browser tab to ws://localhost:8000/ws/positions")

# TODO: replace random data with real skyfield-propagated positions for
# 100 satellites. Build a MapLibre client that consumes the stream.


## What just happened — and why it works

FastAPI's WebSocket support is async-native: each connection is a coroutine. The server pushes JSON arrays at 1 Hz; the client deserializes and updates DOM marker positions. ~100 connected browsers per process is comfortable on commodity hardware.


## Common gotchas

- WebSockets disconnect for many reasons (WiFi flap, server restart, load-balancer hiccup). Production clients MUST reconnect with exponential backoff.
- Browser rendering, not WebSocket throughput, is the bottleneck. 10,000 marker updates / sec will tank any browser.
- CORS rules apply to WebSocket connections. Configure FastAPI's CORS middleware before serving from a different origin.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/19/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
